# Exploratory crime timing analysis

Purpose: compare complete calendar-year crime with crime in the twelve completed months before sale. All specifications and signs are reported. This is exploratory, not a search for a predetermined negative coefficient. Original regressions are unchanged.

Use identical transactions across both exposure measures, all three categories and both geographical specifications. Economic controls exclude job density to retain 2015. Controls include property indicators, continuous transport/river distances, annual pay, hours and unemployment. Compare MSOA + year-quarter with LSOA + year-quarter effects; standard errors cluster by LSOA. Prior crime improves timing but does not eliminate endogeneity.


In [1]:
from pathlib import Path
try:
    from google.colab import drive
except ImportError:
    INPUT_DIR = Path.cwd() / 'project_data'
else:
    drive.mount('/content/drive')
    INPUT_DIR = Path('/content/drive/MyDrive/HE3027 Project/data')
OUT = INPUT_DIR / 'exploratory_crime_timing_outputs'
OUT.mkdir(parents=True, exist_ok=True)


In [2]:
import hashlib
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
CRIMES = ['allcrime', 'violent_crime', 'propertycrime']
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 4)

def table(data, filename):
    """Save the complete table and display a compact preview."""
    data.to_csv(OUT / (filename + '.csv'))
    display(data.head(20) if len(data) > 20 else data)
    if len(data) > 20:
        print(f'Showing 20 of {len(data)} rows; the complete table is saved as {filename}.csv.')

VARIABLE_LABELS = {
    'detached_dum': 'Detached (relative to flat)',
    'semi_d_dum': 'Semi-detached (relative to flat)',
    'terrace_dum': 'Terraced (relative to flat)',
    'freehold': 'Freehold',
    'newbuild': 'New build',
    'tube_distnear_km': 'Distance to tube station (km)',
    'bus_distnear_km': 'Distance to bus stop (km)',
    'thamesriv_dist_km': 'Distance to Thames (km)',
    'Nearest_Park_dist_km': 'Distance to park (km)',
    'dist_to_cbd': 'Distance to CBD (km)',
    'Bus_stopscount_200m': 'Bus stops within 200m',
    'heritage_count_200m': 'Heritage buildings within 200m',
    'percent_noedu': 'Residents without qualifications',
    'percent_minorities': 'Minority residents',
    'lone_hh': 'Lone-parent households',
    'lnpay': 'Log borough annual pay',
    'jobdensity': 'Borough job density',
    'hoursworked': 'Borough weekly hours worked',
    'unemployment': 'Borough unemployment',
}

def coefficient_inference(coefficients, results):
    """Compute inference for every coefficient from its saved clustered SE."""
    keys = [name for name in ['panel', 'model', 'crime', 'measure', 'timing']
            if name in coefficients and name in results]
    metadata = results[keys + ['clusters']]
    enriched = coefficients.drop(columns=['clusters', 't', 'p', 'ci_low', 'ci_high'], errors='ignore')
    enriched = enriched.merge(metadata, on=keys, how='left', validate='many_to_one')
    if enriched.clusters.isna().any():
        raise ValueError('Coefficient rows could not be matched to regression metadata.')
    degrees_freedom = enriched.clusters - 1
    enriched['t'] = enriched.beta / enriched.se
    enriched['p'] = 2 * stats.t.sf(np.abs(enriched.t), degrees_freedom)
    critical = stats.t.ppf(0.975, degrees_freedom)
    enriched['ci_low'] = enriched.beta - critical * enriched.se
    enriched['ci_high'] = enriched.beta + critical * enriched.se
    return enriched

def show_models(results, filename, coefficient_filename):
    """Show one expandable table per model with crime AND control coefficients."""
    from html import escape
    from IPython.display import HTML

    results.to_csv(OUT / (filename + '.csv'), index=False)
    coefficients = coefficient_inference(pd.read_csv(OUT / coefficient_filename), results)
    coefficients.to_csv(OUT / coefficient_filename, index=False)
    cards = []

    def number(value):
        if pd.isna(value):
            return '—'
        if value != 0 and abs(value) < 0.0001:
            return f'{value:.2e}'
        return f'{value:.4f}'

    for _, regression in results.iterrows():
        selected = coefficients.model.eq(regression['model'])
        for key in ['panel', 'crime', 'measure', 'timing']:
            if key in coefficients and key in results:
                selected &= coefficients[key].eq(regression[key])
        rows = coefficients.loc[selected].copy()
        title = regression['model'] + ' · ' + str(regression.get('crime', regression.get('measure', '')))
        if 'timing' in results:
            title += ' · ' + regression['timing']
        rendered = []
        for _, row in rows.iterrows():
            label = VARIABLE_LABELS.get(row.variable, row.variable)
            if row.variable.startswith('log1p_'):
                label = 'Crime: log(1 + ' + row.variable[6:].replace('_', ' ') + ')'
            rendered.append({
                'Variable': label,
                'Coefficient': number(row.beta),
                'Clustered SE': number(row.se),
                't statistic': f'{row.t:.2f}',
                '95% CI': number(row.ci_low) + ' to ' + number(row.ci_high),
                'p-value': '<0.001' if row.p < 0.001 else f'{row.p:.3f}',
            })
        effects = regression.get('fixed_effects', '')
        effects = 'None' if pd.isna(effects) or not effects else str(effects)
        r2_label = 'Within R²' if regression.r2_type == 'within' else 'R²'
        subtitle = (f'N = {int(regression.N):,} | LSOA clusters = {int(regression.clusters):,}'
                    f' | {r2_label} = {regression.r2:.4f}')
        body = pd.DataFrame(rendered).to_html(index=False, border=0, escape=True)
        omitted = regression.get('absorbed_or_collinear', '')
        note = '' if pd.isna(omitted) or not omitted else '<p>Absorbed/collinear: ' + escape(str(omitted)) + '</p>'
        cards.append('<details class="reg-card"><summary>' + escape(title) + '</summary>'
                     '<p>' + escape(subtitle) + '</p><p>Fixed effects: ' + escape(effects) + '</p>'
                     + body + note + '</details>')
    style = '<style>.reg-card{margin:10px 0;padding:12px;border:1px solid #bbb;border-radius:6px;overflow-x:auto}.reg-card summary{cursor:pointer;font-weight:600}.reg-card table{border-collapse:collapse;width:100%;max-width:950px}.reg-card th,.reg-card td{padding:7px 10px;border-bottom:1px solid #ddd;text-align:right;white-space:nowrap}.reg-card th:first-child,.reg-card td:first-child{text-align:left}.reg-card p{font-size:13px}</style>'
    display(HTML(style + '<p>Expand a regression to view all estimated coefficients. Standard errors are clustered by LSOA.</p>' + ''.join(cards)))
    return coefficients

In [3]:
df = pd.read_csv(INPUT_DIR / 'original_crime_models_outputs/land_registry_crime_monthly_merged.csv.gz', low_memory=False)


In [4]:
LOCATION_MODE = 'continuous'
RIVER_THRESHOLD_METRES = None

data = df.copy()
data['lnprice'] = np.log(data.price.where(data.price.gt(0)))
data['newbuild'] = data.age.eq('Y').astype(float)
data['grossannualpay_k'] = data.grossannualpay / 1000
PROPERTY = ['detached_dum','semi_d_dum','terrace_dum','freehold','newbuild']
ECONOMIC = ['grossannualpay_k','jobdensity','hoursworked','unemployment']
if LOCATION_MODE == 'continuous':
    LOCATION = []
    for name in ['tube_distnear','bus_distnear','thamesriv_dist']:
        data[name+'_km'] = data[name]/1000
        LOCATION.append(name+'_km')
elif LOCATION_MODE == 'indicators':
    if RIVER_THRESHOLD_METRES is None or RIVER_THRESHOLD_METRES <= 0:
        raise ValueError('Specify a positive river proximity threshold in metres.')
    thresholds = {'tube_distnear':200, 'bus_distnear':200, 'thamesriv_dist':RIVER_THRESHOLD_METRES}
    LOCATION = ['tube_near','bus_near','river_near']
    for name, target in zip(thresholds,LOCATION):
        data[target] = data[name].le(thresholds[name]).astype(float).where(data[name].notna())
else:
    raise ValueError('LOCATION_MODE must be continuous or indicators.')
VARIABLE_LABELS.update({'grossannualpay_k':'Borough annual pay (GBP 1,000)',
                        'tube_near':'Tube station within 200m', 'bus_near':'Bus stop within 200m',
                        'river_near':'River within specified threshold'})

In [5]:
def absorb(matrix, groups, tol=1e-10, max_iter=2000):
    z = np.asarray(matrix, dtype=float).copy()
    codes = [pd.factorize(g, sort=False)[0] for g in groups]
    if any(((g < 0).any() for g in codes)):
        raise ValueError('Missing fixed-effect ID')
    for iteration in range(max_iter):
        before = z.copy()
        for g in codes:
            counts = np.bincount(g)
            sums = np.column_stack([np.bincount(g, weights=z[:, j], minlength=len(counts)) for j in range(z.shape[1])])
            z -= (sums / counts[:, None])[g]
        if np.max(np.abs(z - before)) < tol:
            return (z, iteration + 1)
    raise RuntimeError('Fixed-effect absorption did not converge')

In [6]:
def fit_absorbed(data, crime, controls, fixed_effects, cluster='lsoacode'):
    names = ([crime] if crime is not None else []) + controls
    x = data[names].to_numpy(dtype=float, copy=True)
    scale = x.std(axis=0)
    scale[scale < 1e-14] = 1
    x /= scale
    y = data.lnprice.to_numpy(float)
    if fixed_effects:
        z, iterations = absorb(np.column_stack([y, x]), [data[c] for c in fixed_effects])
        y, x = (z[:, 0], z[:, 1:])
    else:
        y = y - y.mean()
        x = x - x.mean(axis=0)
        iterations = 0
    gram = x.T @ x
    keep = []
    for j in range(x.shape[1]):
        residual_norm2 = gram[j, j]
        if keep:
            cross = gram[keep, j]
            residual_norm2 -= cross @ np.linalg.solve(gram[np.ix_(keep, keep)], cross)
        if residual_norm2 > 1e-10 * len(data):
            keep.append(j)
    if crime is not None and 0 not in keep:
        raise ValueError('Crime has no identifying variation in this specification')
    design = x[:, keep]
    beta = np.linalg.lstsq(design, y, rcond=None)[0]
    residual = y - design @ beta
    group = pd.factorize(data[cluster], sort=False)[0]
    G = group.max() + 1
    if G < 2:
        raise ValueError('Need multiple inference clusters')
    scores = np.zeros((G, len(keep)))
    for j in range(len(keep)):
        scores[:, j] = np.bincount(group, weights=design[:, j] * residual, minlength=G)
    bread = np.linalg.inv(design.T @ design)
    covariance = bread @ (scores.T @ scores) @ bread * G / (G - 1)
    estimates = beta / scale[keep]
    se = np.sqrt(np.maximum(np.diag(covariance), 0)) / scale[keep]
    j = keep.index(0)
    critical = stats.t.ppf(0.975, G - 1)
    result = {'N': len(data), 'clusters': G, 'beta': estimates[j], 'se': se[j], 'p': 2 * stats.t.sf(abs(estimates[j] / se[j]), G - 1), 'ci_low': estimates[j] - critical * se[j], 'ci_high': estimates[j] + critical * se[j], 'r2': 1 - residual @ residual / (y @ y), 'r2_type': 'within' if fixed_effects else 'ordinary', 'crime_residual_sd': x[:, 0].std() * scale[0], 'iterations': iterations, 'absorbed_or_collinear': '; '.join((names[j] for j in range(len(names)) if j not in keep)), 'fixed_effects': '; '.join(fixed_effects)}
    coefficients = pd.DataFrame({'variable': [names[j] for j in keep], 'beta': estimates, 'se': se})
    if crime is None:
        for key in ['beta','se','p','ci_low','ci_high','crime_residual_sd']:
            result[key] = np.nan
    return (result, coefficients)

In [7]:
MAIN = ['mps_all', 'mps_violence', 'mps_property']
controls = PROPERTY + LOCATION + [c for c in ECONOMIC if c != 'jobdensity']
crime_columns=[]
for category in MAIN:
    for exposure in ['annual','prior12']:
        name='log1p_'+category+'_'+exposure
        data[name]=np.log1p(data[category+'_'+exposure])
        crime_columns.append(name)
required=['lnprice','lsoacode','msoa11','yearquarter','oslaua']+controls+crime_columns
valid=data[required].replace('',np.nan).notna().all(axis=1)
valid &= np.isfinite(data[required].select_dtypes(include='number')).all(axis=1)
valid &= data.mps_geography_ok & data.oslaua.str.startswith('E090',na=False)
borough_count=data.loc[data.oslaua.str.startswith('E090',na=False)].groupby('lsoacode').oslaua.nunique()
valid &= ~data.lsoacode.isin(borough_count.index[borough_count.gt(1)])
sample=data.loc[valid].copy()
print('Common sample:',len(sample),'sales;',sample.lsoacode.nunique(),'LSOAs')
table(sample.groupby('year').size().to_frame('sales'),'common_sample_by_year')


Common sample: 405088 sales; 4648 LSOAs


,sales
year,
2011,66741
2012,85375
2013,99429
2014,108207
2015,45336


In [8]:
rows=[]
coefficients=[]
for geography in ['msoa11','lsoacode']:
    for exposure in ['annual','prior12']:
        for category in MAIN:
            crime='log1p_'+category+'_'+exposure
            result,coef=fit_absorbed(sample,crime,controls,[geography,'yearquarter'])
            result.update(geography=geography,exposure=exposure,crime=category)
            rows.append(result)
            coefficients.append(coef.assign(geography=geography,exposure=exposure,crime=category))
            print(geography,exposure,category,'beta=',round(result['beta'],5),'p=',round(result['p'],4))
results=pd.DataFrame(rows)
assert len(results)==12 and results.N.nunique()==1
results.to_csv(OUT/'all_exploratory_models.csv',index=False)
pd.concat(coefficients).to_csv(OUT/'all_exploratory_coefficients.csv',index=False)
table(results[['geography','exposure','crime','N','beta','se','ci_low','ci_high','p']],'crime_coefficients')


msoa11 annual mps_all beta= -0.01019 p= 0.0369
msoa11 annual mps_violence beta= -0.05053 p= 0.0
msoa11 annual mps_property beta= 0.00996 p= 0.0308
msoa11 prior12 mps_all beta= -0.00987 p= 0.0452
msoa11 prior12 mps_violence beta= -0.04952 p= 0.0
msoa11 prior12 mps_property beta= 0.0096 p= 0.0393
lsoacode annual mps_all beta= 0.00658 p= 0.1601
lsoacode annual mps_violence beta= -0.00257 p= 0.3133
lsoacode annual mps_property beta= 0.01069 p= 0.0067
lsoacode prior12 mps_all beta= 0.00171 p= 0.7477
lsoacode prior12 mps_violence beta= -0.00275 p= 0.303
lsoacode prior12 mps_property beta= 0.00574 p= 0.1797


,geography,exposure,crime,N,beta,se,ci_low,ci_high,p
0,msoa11,annual,mps_all,405088,-0.0102,0.0049,-0.0198,-0.0006,3.6876e-02
1,msoa11,annual,mps_violence,405088,-0.0505,0.0039,-0.0582,-0.0428,2.3418e-37
2,msoa11,annual,mps_property,405088,0.0100,0.0046,0.0009,0.0190,3.0822e-02
3,msoa11,prior12,mps_all,405088,-0.0099,0.0049,-0.0195,-0.0002,4.5166e-02
4,msoa11,prior12,mps_violence,405088,-0.0495,0.0039,-0.0571,-0.0419,9.4059e-37
5,msoa11,prior12,mps_property,405088,0.0096,0.0047,0.0005,0.0187,3.9348e-02
6,lsoacode,annual,mps_all,405088,0.0066,0.0047,-0.0026,0.0158,1.6006e-01
7,lsoacode,annual,mps_violence,405088,-0.0026,0.0026,-0.0076,0.0024,3.1331e-01
8,lsoacode,annual,mps_property,405088,0.0107,0.0039,0.0030,0.0184,6.6548e-03
9,lsoacode,prior12,mps_all,405088,0.0017,0.0053,-0.0087,0.0121,7.4770e-01


## Interpretation limits

Both exposure measures are estimated on the same transactions to distinguish timing differences from sample differences. Annual counts can include incidents after a sale; prior12 counts exclude the sale month and all later incidents. Neither controls for time-varying LSOA confounding, unobserved property quality or reporting changes. Results are exploratory and every estimate is retained. The estimator applies a cluster-count correction without an additional absorbed-effect degrees-of-freedom correction.


## Findings from this exploration

All 12 regressions use the same 405,088 sales in 4,648 LSOAs. With prior-12-month crime and MSOA plus year-quarter effects, all crime is negative (-0.00987, p=0.045), violence is negative (-0.04952, p<0.001), and property crime is positive (+0.00960, p=0.039). With LSOA plus year-quarter effects, the corresponding estimates are +0.00171 (p=0.748), -0.00275 (p=0.303), and +0.00574 (p=0.180); all three confidence intervals include zero.

The timing improvement does not establish a consistent negative within-LSOA relationship. The negative MSOA associations should be reported alongside their sensitivity to finer geographical effects. Selecting the MSOA model solely because its coefficient is negative would conceal an important limitation. A defensible report can examine how the association changes with geographical controls and why the evidence does not support a precise causal valuation.
